# Moving Averages

As the price moves up and down we will want to estimate the optimum price with a moving average. Below, we consider trailing Window moving average and Trailing Exponential moving average.

Note - we keep this section in the notebook for expository reasons, though we are not making use of these moving averages in the notebook. The overview and discussion below are useful.

### Window moving average vs Exponential moving average



Rule of thumb relationship

$$
\alpha \approx \frac{2}{Nwindow + 1} 
$$

or  equivalently

$$
 Nwindow \approx 2/\alpha - 1
$$  

* Large α → reacts quickly to recent changes (less smoothing).
* Small α → reacts slowly (more smoothing).
* Large window → more smoothing.
* Small window → less smoothing.

&nbsp;

| Window (`Nwindow`) | Equivalent `alpha` |
|-------------------:|-------------------:|
| 3                  | 0.50               |
| 5                  | 0.33               |
| 10                 | 0.18               |
| 20                 | 0.095              |
| 50                 | 0.039              |


The Lag, corresponding to tthe exponential MA, gives a percentage of the previous row. A lag of 0.3 each row is (1 -$\alpha$) of the previous row. If lag is 0.3 then each row is 70% of the previous row. 

Lag  with alpha = 0.33

| Lag | Weight Formula | Weight (α = 0.33) |
|----:|----------------|------------------:|
| 0 | $\alpha(1-\alpha)^0$ | 0.3300 |
| 1 | $\alpha(1-\alpha)^1$ | 0.2211 |
| 2 | $\alpha(1-\alpha)^2$ | 0.1481 |
| 3 | $\alpha(1-\alpha)^3$ | 0.0992 |
| 4 | $\alpha(1-\alpha)^4$ | 0.0664 |
| 5 | $\alpha(1-\alpha)^5$ | 0.0445 |
| 6 | $\alpha(1-\alpha)^6$ | 0.0298 |
| 7 | $\alpha(1-\alpha)^7$ | 0.0200 |
| 8 | $\alpha(1-\alpha)^8$ | 0.0134 |
| 9 | $\alpha(1-\alpha)^9$ | 0.0090 |

from moving_averages_trailing import trailing_exponential_moving_average

# test
prices = [3, 4, 5, 6, 7, 8, 9]

trailing_exponential_moving_average(prices, alpha=0.33)[-1]

from moving_averages_trailing import trailing_moving_average

# test
prices = [3, 4, 5, 6, 7, 8, 9]

trailing_moving_average(prices, Nwindow=3)[-1]

Based on the previous two examples, I prefer the trailing window MA. Notice that for the progression of 5, 6, 7, 8 , 9, the trailing window gives an an averate of 8 while exponential moving average gives an average of 7.15, thus the lag is more prominant. 

# Demand Fit Theory

Estimating elasticity in the case of demand shocks is deceptibly difficut. Though it would weem to be a simple regression there are some complications. 

* Physical interpretation of elasticity where going below elasticity can make it look like elasticity switches between an elastic demand to an inelastic demand. 
* Noisy demands on successive differences can elasticity look negative with unrealistic absolute values.


### Model Choices

Model choices include
* Level-linear: $d = \beta_0 + \beta_1 \, p_1$
* Log-level: log \, $d = \beta_0 + \beta_1 \, p $
* Log-log: log \, $d = \beta_0 + \beta_1 \, log \, p $
* Log-quadratic: log \, $d = \beta_0  + \beta_1 \, log \, p + \beta_2 (log \, p)^2$


Experiments
* Log-quadratic: We began with a Log-quadratic model. We find that it fits logit demand (with noisy demand) well. It also does a reasonable job with constant elasticy demand. However, though it still works, it struggles a bit more with linear demand

Elasticity Fit (log-quadratic)

- Quadratic regression in log-space
- log d = $\beta_0 + \beta_1 \log p + \beta_2 (\log p)^2 + \gamma x + e$
- vhat(p) = -($\beta_1 + 2 \beta_2 \log p $)

In this fit, the parameters are:

- $\beta_0 $: y-intercept
- $\beta_1$: slope or linear coefficient
- $\beta_2$: quadratic curvature
- $\gamma$: multiplier of endogenous features
- $e$: random error on the log scale

Approach

- Use all historical observations to smooth out demand shocks and estimate the demand curve.
- Evaluate the local slope and elasticity at the estimated optimum price (p_opt_est)

Fitting a smooth curve reduces random demand noise, but it cannot remove bias when demand shocks systematically coincide with price movements.

Example of Contextual Variables x
* traffic
* competitor price
* promotion
* weekend

x with 10 observations

```text
x = np.array([
    [10500, 2.55, 0, 0],
    [10200, 2.55, 0, 0],
    [11000, 2.60, 1, 0],
    [10800, 2.60, 1, 0],
    [ 9800, 2.65, 0, 0],
    [12000, 2.70, 0, 1],
    [11800, 2.70, 0, 1],
    [11500, 2.75, 0, 0],
    [11200, 2.80, 0, 0],
    [11700, 2.85, 1, 1],
])
```

**Demand Fit**

Without context features, x

```python
dhat, model_info = fit_demand_estimator(
    p=prices,
    d=demand,
    x=None,
)
```